# Gradient/bval signal ANALYSIS for FF_dwi_drift
------------------------------------------------------

1) The first idea is to identify signal drift in the acuqired signal - In order to do it, we will run several GLM analysis in each gradient, hopefully we will find significant effect of gradients on spacial signal. 
b-value: the .bval file gives the overall diffusion weighting for that volume, from the Stejskal-Tanner relation: b = γ²G²δ²(Δ − δ/3) where G is gradient amplitude, δ is pulse duration, and Δ is the time between the two diffusion gradient lobes.
    b = γ²G²δ²(Δ − δ/3): [i asked Claude]
        
        γ = 2.675×10⁸ rad/s/T (water protons)
        δ (pulse duration)	~20 ms	
        Δ (lobe separation)	~35 ms	


2) Separate each bval --> Average volume * each bval then compute a GLM between ses-01 and ses-02 on the merged sequence. 

## Separate Shells - ABC merged seq. 
------------------------------------------

In [ ]:
## Same code but in parallel (using joblib) to speed up the processing of multiple subjects/sessions.
import os
import glob
import subprocess as sp
import numpy as np
import nibabel as nib
from nilearn import image as nlimage
from concurrent.futures import ProcessPoolExecutor

import warnings
warnings.filterwarnings("ignore")

# ===============================================
# Configuration
# ===============================================
vps      = [i for i in range(44) if i not in [32]]   # list of subject IDs
sessions = ["ses-02"]

home = r"/home/malberti/Unix_Folders/SWEEP2/derivatives"

TEMPLATE = "MNI"   # "HCPex" or "MNI"
TEMPLATE_PATHS = {
    "HCPex": r"/home/malberti/Unix_Folders/SWEEP2/Script/DEWEY_v6/Atlases/MNI_icbm_152_Template\tpl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz",   # <-- fill in
    "MNI":   r"/usr/local/fsl/data/standard/MNI152_T1_2mm_brain.nii.gz" ,  # <-- fill in
}
TEMPLATE_PATH = TEMPLATE_PATHS[TEMPLATE]

BVAL_TOLERANCE = 50    # b-values within +/-50 s/mm^2 of a shell centre are grouped together
SMOOTH_FWHM_MM = 6     # final smoothing kernel

N_WORKERS = 3   # <-- adjust to available cores


# ===============================================
# Helpers
# ===============================================
def EXTRACT_SHELLs(DWIs, BVALs, SHELL, out_path):
    """
    Extract the volumes at `indices` from a 4D DWI file and average them
    across directions, producing one 3D volume for that shell.
    """
    
    dwi  = nib.load(DWIs) # Load the 4D DWI image
    dwi_data = dwi.get_fdata()
    indices = np.where(np.abs(BVALs - SHELL) <= BVAL_TOLERANCE)[0]
   
    print(f"[INFO] Extracting shell b={SHELL} with {len(indices)} volumes")
    
    shell_data = dwi_data[..., indices]
    SHELL_mean = np.mean(shell_data, axis=-1)  # Average across directions
    SHELLS_out = nib.Nifti1Image(SHELL_mean, affine=dwi.affine, header=dwi.header)
    nib.save(SHELLS_out, out_path)
    
    return out_path


def COREG_to_MNI(native_vol, out_path, transform_nonlin, transform_lin, template_path=TEMPLATE_PATH):
    """
    Apply an existing native -> template transform to a single 3D volume.
    NOTE: antsApplyTransforms uses -i (moving/input image) and -r (fixed/
    reference image), not -f/-m (those are antsRegistration flags).
    """
    print(f"[INFO] Coregistering {native_vol} to template {template_path} using transforms {transform_nonlin} and {transform_lin}")
    cmd = [
        "antsApplyTransforms", "-d", "3",
        "-i", native_vol,
        "-r", template_path,
        "-o", out_path,
        "-t", transform_nonlin,
        "-t", transform_lin,
        "--interpolation", "Linear",
    ]
    sp.run(cmd, check=True)
    return out_path


def process_subject_session(vp, ses):
    pid    = f"sub-{vp:02d}"
    subjid = f"{pid}_{ses}"

    derivatives = os.path.join(home, pid, ses, "dwi")
    DWIs=os.path.join(derivatives,'signal_drift', f"{subjid}_eddy-current_signal-drift_ABC-2T1w_corr.nii.gz")
    BVALs = np.loadtxt(os.path.join(derivatives,'eddy', f"{subjid}_dwi_eddy_corrected_ABC_noPA2T1w.bval"))
    anat = os.path.join(home, pid, "ses-02", "anat")

    t1w2MNI=os.path.join(anat,"mni-registration", f"{pid}_ses-02_desc-nonlin1warp_xfm.nii.gz") #Non sub-200_ses-02_desc-nonlin1warp_xfm linear warp 
    t1w2MNI_lin=os.path.join(anat,"mni-registration", f"{pid}_ses-02_desc-nonlin0genericaffine_xfm.mat" )

    # Group b-values into shells within +/- BVAL_TOLERANCE of a rounded centre
    SHELLs=np.unique(np.round(BVALs / BVAL_TOLERANCE) * BVAL_TOLERANCE)

    print(f"[INFO] Processing {subjid}: {len(SHELLs)} shells found: {SHELLs}")
    
    # Outdir
    out_dir = os.path.join(derivatives, pid, ses, "dwi", "Signal_Stability", "tmp")
    os.makedirs(out_dir, exist_ok=True)
    
    SHELLs_out_paths = []
    for SHELL in SHELLs:
        out_shell = os.path.join(out_dir, f"{subjid}_b{str(SHELL)}.nii.gz")
        EXTRACT_SHELLs(DWIs, BVALs, SHELL, out_shell) #Extract the shells and average across volumes/time
        
        SHELLs_out_paths.append(out_shell) #Get the paths of the extracted shells
    
    for i,file in enumerate(SHELLs_out_paths):
        out_coreg = os.path.join(out_dir, f"{subjid}_b{str(SHELLs[i])}-2{TEMPLATE}.nii.gz")
        COREG_to_MNI(file, out_coreg, t1w2MNI, t1w2MNI_lin)
        
        #Smooths
        smooth_path = os.path.join(out_dir, f"{subjid}_b{str(SHELLs[i])}-2{TEMPLATE}{SMOOTH_FWHM_MM}.nii.gz")
        smoothed = nlimage.smooth_img(out_coreg, fwhm=SMOOTH_FWHM_MM)
        smoothed.to_filename(smooth_path)


# ===============================================
# STAGE 1 — per subject/session: extract shells, average directions, coreg
# ===============================================
if __name__ == "__main__":
    jobs = [(vp, ses) for vp in vps for ses in sessions]
    with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
        futures = {ex.submit(process_subject_session, vp, ses): (vp, ses) for vp, ses in jobs}
        for f in futures:
            vp, ses = futures[f]
            try:
                f.result()
            except Exception as e:
                print(f"[ERROR] sub-{vp:02d} {ses} failed: {e}")

## Run GLM on each shell and plots the results on glass brain | POTENTIALLY WRONG
-----------------------------------------------------------------------

I stole the code directly from out previous analysis 

In [ ]:
# Compute matrix 
# ========================
# === CONFIGURATION    ===
# ========================
from scipy.__config__ import show
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
from nilearn.plotting import plot_design_matrix, plot_glass_brain, show, plot_stat_map
from nilearn.glm.second_level import SecondLevelModel
from scipy.stats import norm
import nibabel as nib
import subprocess
from nilearn import datasets
from nilearn.glm import threshold_stats_img
from nilearn import plotting

# In this example we will plot both hemispheres, but you can choose one of
# "left", "right" or "both".
from nilearn.datasets import load_fsaverage_data


sessions = ["ses-01", "ses-02"]
APs     = ["ABC"]
prj_path=r"/home/malberti/Unix_Folders/SWEEP2" # main BIDS path /home/malberti/Unix_Folders/Sweep/Pilot/SW005
prj_data = r"/home/malberti/wks14/temp/SWEEP/malberti/SWEEP2/physio/physio_psyco_measure_v2.ods" #physiological data
exclude  = ["sub-32"]

# ========================
# === LOAD DATA        ===
# ========================
Stress_Placebo = pd.read_excel(prj_data, sheet_name='Stress-Placebo', engine="odf")

# Filter out excluded subjects
Stress_Placebo = Stress_Placebo[~Stress_Placebo["Sub-ID"].isin(exclude)]

# Now create lists
vps = Stress_Placebo["Sub-ID"].tolist()
stress = Stress_Placebo["Stress "].tolist()
placebo = Stress_Placebo["Placebo"].tolist()

# Create design and contrast matrix
n_subjects = len(vps)


outdir = os.path.join(prj_path, 'Statistical_analysis')
os.makedirs(outdir, exist_ok=True)

## Nilearn matrix for paired t-test - from Nilearn tutorial  https://nilearn.github.io/dev/auto_examples/05_glm_second_level/plot_second_level_two_sample_test.html

condition_effect = np.hstack(([1] * n_subjects, [0] * n_subjects))
subject_effect = np.vstack((np.eye(n_subjects), np.eye(n_subjects)))
subjects = [f"S{i:02d}" for i in range(1, n_subjects + 1)]

paired_design_matrix = pd.DataFrame(
    np.hstack((condition_effect[:, np.newaxis], subject_effect)),
    columns=["Ses-01_ws_Ses-02", *subjects],
)

fig, ax = plt.subplots(1, 1, figsize=(8, 6), constrained_layout=True)
plot_design_matrix(paired_design_matrix, rescale=False, axes=ax)
ax.set_title("Paired design", fontsize=12)

#show()[1] * n_subjects, [0]

## ================================
vps      = [i for i in range(44) if i not in [32]]   # list of subject IDs
sessions = ["ses-01", "ses-02"]

SMOOTH_FWHM_MM = "6"
TEMPLATE = "MNI"
BVAL_TOLERANCE = 20
# Shells are the same across subjects/sessions, so get them once
pid = f"sub-{vps[0]:02d}"
derivatives0 = os.path.join(home, pid, "ses-01", "dwi")
BVALs0 = np.loadtxt(os.path.join(derivatives0, 'eddy', f"{pid}_ses-01_dwi_eddy_corrected_ABC_noPA2T1w.bval"))
SHELLs = np.unique(np.round(BVALs0 / BVAL_TOLERANCE) * BVAL_TOLERANCE)

for SHELL in SHELLs:
    ses01_IMGs = []
    ses02_IMGs = []

    for vp in vps:
        pid = f"sub-{vp:02d}"

        ### ________________________ 1ST session _____________________
        subjid = f"{pid}_ses-01"
        derivatives = os.path.join(prj_path, "derivatives", pid, "ses-01", "dwi", pid, "ses-01", "dwi", "Signal_Stability", "tmp")
        ses01_IMGs.append(os.path.join(derivatives, f"{subjid}_b{str(SHELL)}-2{TEMPLATE}{SMOOTH_FWHM_MM}.nii.gz"))

        ### ________________________ 2ND session _____________________
        subjid = f"{pid}_ses-02"
        derivatives = os.path.join(prj_path, "derivatives", pid, "ses-02", "dwi", pid, "ses-02", "dwi", "Signal_Stability", "tmp")
        ses02_IMGs.append(os.path.join(derivatives, f"{subjid}_b{str(SHELL)}-2{TEMPLATE}{SMOOTH_FWHM_MM}.nii.gz"))

    Ses_01_ws_Ses_02 = ses01_IMGs + ses02_IMGs
    print(Ses_01_ws_Ses_02)

    Ses_01_ws_Ses_02_paired = SecondLevelModel(n_jobs=2, verbose=0).fit(Ses_01_ws_Ses_02, design_matrix=paired_design_matrix)
    MAPs_Ses_01_ws_Ses_02 = Ses_01_ws_Ses_02_paired.compute_contrast("Ses-01_ws_Ses-02", output_type="all")

    z_score = MAPs_Ses_01_ws_Ses_02["z_score"]
    thresholded_map1, threshold1 = threshold_stats_img(z_score, alpha=0.001, cluster_threshold=10, two_sided=True) #, height_control="fdr")

    fig = plotting.plot_glass_brain(
        thresholded_map1,
        threshold=3.29,
        colorbar=True,
        figure=plt.figure(figsize=(12, 6), dpi=150),
        vmax=5.8,
        vmin=-5.8,
        symmetric_cbar=True,
        cmap="bwr",
        draw_cross=False,
        display_mode="ortho",  # l, r, z projections
        title=f"Ses-01 vs Ses-02: b={SHELL} - unc.p - ALL.",
    )
    show()

In [ ]:
pip install odfpy

# Gradients directionaity: 
-----------------------------------------
The followings blocks want to check wheter the gradient directions has something to do with the artifacts, to do it, i selected the same direction from each participant, moved them to MNI +6mm smoothing, then i compute a GLM and finally build a 1-pval time series 

In [ ]:
## Same code but in parallel (using joblib) to speed up the processing of multiple subjects/sessions.
import os
import glob
import subprocess as sp
import numpy as np
import nibabel as nib
from nilearn import image as nlimage
from concurrent.futures import ProcessPoolExecutor

import warnings
warnings.filterwarnings("ignore")

# ===============================================
# Configuration
# ===============================================
vps      = 0 # [i for i in range(43) if i not in [32]]   # list of subject IDs
sessions = [ "ses-02"]  #"ses-01",

#os.chdir(os.path.join(script, 'DEWEY_v6'))
#Change acqparams and index file accordingly to 
vp_ABC=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 39, 40, 41, 43]
vp_BCA=[18, 19, 20, 21, 23, 24, 22, 25, 26, 27, 28, 29, 30, 31, 33, 34, 35, 36, 37, 38, 42]

home = r"/home/malberti/Unix_Folders/SWEEP2/derivatives"

TEMPLATE = "MNI"   # "HCPex" or "MNI"
TEMPLATE_PATHS = {
    "HCPex": r"/home/malberti/Unix_Folders/SWEEP2/Script/DEWEY_v6/Atlases/MNI_icbm_152_Template\tpl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz",   # <-- fill in
    "MNI":   r"/usr/local/fsl/data/standard/MNI152_T1_2mm_brain.nii.gz" ,  # <-- fill in
}
TEMPLATE_PATH = TEMPLATE_PATHS[TEMPLATE]

BVAL_TOLERANCE = 50    # b-values within +/-50 s/mm^2 of a shell centre are grouped together
SMOOTH_FWHM_MM = 6     # final smoothing kernel

N_WORKERS = 3   # <-- adjust to available cores

APs=["B", "C"] #"A",

# ===============================================
# Helpers
# ===============================================
def EXTRACT_SHELLs(DWIs, BVALs, SHELL, out_path):
    """
    Extract the volumes at `indices` from a 4D DWI file and average them
    across directions, producing one 3D volume for that shell.
    """
    
    dwi  = nib.load(DWIs) # Load the 4D DWI image
    dwi_data = dwi.get_fdata()
    indices = np.where(np.abs(BVALs - SHELL) <= BVAL_TOLERANCE)[0]
   
    print(f"[INFO] Extracting shell b={SHELL} with {len(indices)} volumes")
    
    shell_data = dwi_data[..., indices]
    SHELL_mean = np.mean(shell_data, axis=-1)  # Average across directions
    SHELLS_out = nib.Nifti1Image(SHELL_mean, affine=dwi.affine, header=dwi.header)
    nib.save(SHELLS_out, out_path)
    
    return out_path


def COREG_to_MNI(native_vol, out_path, transform_nonlin, transform_lin, template_path=TEMPLATE_PATH):
    """
    Apply an existing native -> template transform to a single 3D volume.
    NOTE: antsApplyTransforms uses -i (moving/input image) and -r (fixed/
    reference image), not -f/-m (those are antsRegistration flags).
    """
    print(f"[INFO] Coregistering {native_vol} to template {template_path} using transforms {transform_nonlin} and {transform_lin}")
    cmd = [
        "antsApplyTransforms", "-d", "3",
        "-i", native_vol,
        "-r", template_path,
        "-o", out_path,
        "-t", transform_nonlin,
        "-t", transform_lin,
        "--interpolation", "Linear",
    ]
    sp.run(cmd, check=True)
    return out_path

In [ ]:
import os
import numpy as np
import nibabel as nib
from itertools import product
import tempfile
import subprocess
from concurrent.futures import ProcessPoolExecutor

def process_ses_ap(ses, AP, vps, home, out_dir):
    dictionary = {}
    n_volumes = None
    vp_BCA=[18, 19, 20, 21, 23, 24, 22, 25, 26, 27, 28, 29, 30, 31, 33, 34, 35, 36, 37, 38, 42]

    for vp in vps:
        AP_alt = {"A": "B", "B": "C", "C": "A"}[AP] if vp in vp_BCA else AP
        pid = f"sub-{vp:02d}"
        subjid = f"{pid}_{ses}"
        path = os.path.join(home, pid, ses, "dwi", "signal_drift", f"{subjid}_dwi_eddy_corrected_{AP_alt}_noPA.nii")
        dictionary[vp] = path
        if n_volumes is None:
            n_volumes = nib.load(path).shape[-1]

    print(f"[INFO] Stacking data for {ses} {AP}")
    with tempfile.TemporaryDirectory() as tmp:
        for vol in range(n_volumes):
            vol_files = []
            for vp in vps:
                vol_path = os.path.join(tmp, f"vp{vp}_vol{vol}.nii.gz")
                subprocess.run(["fslroi", dictionary[vp], vol_path, str(vol), "1"], check=True)
                vol_files.append(vol_path)
            out_path = os.path.join(out_dir, f"{ses}_{AP}_vol{vol}.nii.gz")
            subprocess.run(["fslmerge", "-t", out_path] + vol_files, check=True)
            for f in vol_files:
                os.remove(f)

    print(f"[INFO] Done {ses} {AP}")
    return f"{ses}_{AP}"


if __name__ == "__main__":
    out_dir = os.path.join("/home/malberti/Unix_Folders/SWEEP2", "Gradients_Stability")
    os.makedirs(out_dir, exist_ok=True)

    AP_session = list(product(sessions, APs))
    with ProcessPoolExecutor(max_workers=50) as executor:
        futures = [executor.submit(process_ses_ap, ses, AP, vps, home, out_dir)for ses, AP in AP_session]
        for f in futures:
            print(f"[INFO] Finished {f.result()}")

### The previous block created the volume time series, now, same volumes series should be merged and coregistered to MNI space + 6mm smoothing. 
-----------------------------------------------------------------------------------------------------------------------------------------------------

# Coreg. Volumes to MNI space before srting them by volumes
---------------------------------------------------------------------------------------------------

In [ ]:
import subprocess
import os
import numpy as np
import nibabel as nib
from itertools import product
from concurrent.futures import ProcessPoolExecutor

import warnings
warnings.filterwarnings("ignore")

# ===============================================
# Configuration
# ===============================================
vp_ABC = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 39, 40, 41, 43]
vp_BCA = [18, 19, 20, 21, 23, 24, 22, 25, 26, 27, 28, 29, 30, 31, 33, 34, 35, 36, 37, 38, 42]
vps = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 39, 40, 41, 43, 18, 19, 20, 21, 23, 24, 22, 25, 26, 27, 28, 29, 30, 31, 33, 34, 35, 36, 37, 38, 42]

sessions = ["ses-01", "ses-02"]
APs = ["A", "B", "C"]

home = r"/home/malberti/Unix_Folders/SWEEP2/derivatives"

TEMPLATE = "MNI"
TEMPLATE_PATH = r"/usr/local/fsl/data/standard/MNI152_T1_2mm_brain.nii.gz"

out_dir = os.path.join("/home/malberti/Unix_Folders/SWEEP2", "Gradients_Stability_analysis")
os.makedirs(out_dir, exist_ok=True)

N_WORKERS = 7


def get_AP_alt(vp, AP_bca):
    if vp in vp_BCA:
        return {"A": "B", "B": "C", "C": "A"}[AP_bca]
    return AP_bca

def build_transforms(vp, ses, AP):
    pid = f"sub-{vp:02d}"
    subjid = f"{pid}_{ses}"
    derivatives = os.path.join(home, pid, ses, "dwi")
    anat = os.path.join(home, pid, "ses-02", "anat")
    return {
        "FA2t1w_matrix": os.path.join(derivatives, "coreg", f"{subjid}_{AP}_FA2t1w_0GenericAffine.mat"),
        "t1w2MNI":       os.path.join(anat, "mni-registration", f"{pid}_ses-02_desc-nonlin1warp_xfm.nii.gz"),
        "t1w2MNI_lin":   os.path.join(anat, "mni-registration", f"{pid}_ses-02_desc-nonlin0genericaffine_xfm.mat"),
    }

def register_4d_to_mni(vp, ses, AP, template_path=TEMPLATE_PATH):
    AP_alt = get_AP_alt(vp, AP)
    pid = f"sub-{vp:02d}"
    subjid = f"{pid}_{ses}"

    native_4d = os.path.join(home, pid, ses, "dwi", "signal_drift", f"{subjid}_eddy-current_signal-drift_{AP}_corr.nii.gz")
    out_path = os.path.join(out_dir, f"{subjid}_{AP_alt}_desc-{TEMPLATE}.nii.gz")
    tfms = build_transforms(vp, ses, AP)

    print(f"[INFO] Registering {native_4d} -> {template_path}")
    cmd = [
        "antsApplyTransforms", "-d", "3", "-e", "3",
        "-i", native_4d,
        "-r", template_path,
        "-o", out_path,
        "-t", tfms["t1w2MNI"],
        "-t", tfms["t1w2MNI_lin"],
        "-t", tfms["FA2t1w_matrix"],
        "--interpolation", "BSpline",
    ]
    subprocess.run(cmd, check=True)
    print(f"[INFO] Done {subjid} {AP}")
    return out_path

# ===============================================
# Main
# ===============================================
if __name__ == "__main__":
    jobs = list(product(vps, sessions, APs))
    with ProcessPoolExecutor(max_workers=N_WORKERS) as executor:
        futures = [executor.submit(register_4d_to_mni, vp, ses, AP) for vp, ses, AP in jobs]
        for f in futures:
            print(f"[INFO] Finished {f.result()}")

In [ ]:
import os
import nibabel as nib
from itertools import product
import tempfile
import subprocess
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

def process_ses_ap(ses, AP, vps, mni_dir, out_dir):
    dictionary = {}
    n_volumes = None

    for vp in vps:
        pid = f"sub-{vp:02d}"
        subjid = f"{pid}_{ses}"
        path = os.path.join(mni_dir, f"{subjid}_{AP}_desc-MNI.nii.gz")
        dictionary[vp] = path
        if n_volumes is None:
            n_volumes = nib.load(path).shape[-1]

    print(f"[INFO] Stacking data for {ses} {AP}")
    with tempfile.TemporaryDirectory() as tmp:
        for vol in range(n_volumes):
            vol_files = []
            for vp in vps:
                vol_path = os.path.join(tmp, f"vp{vp}_vol{vol}.nii.gz")
                subprocess.run(["fslroi", dictionary[vp], vol_path, str(vol), "1"], check=True)
                print(vol_path)
                vol_files.append(vol_path)
            out_path = os.path.join(out_dir, f"{ses}_{AP}_vol{vol}.nii.gz")
            subprocess.run(["fslmerge", "-t", out_path] + vol_files, check=True)
          #  for f in vol_files:
          #     os.remove(f)

    print(f"[INFO] Done {ses} {AP}")
    return f"{ses}_{AP}"

if __name__ == "__main__":
    mni_dir = os.path.join("/home/malberti/Unix_Folders/SWEEP2", "Gradients_Stability_analysis")
    out_dir = os.path.join("/home/malberti/Unix_Folders/SWEEP2", "Gradients_Stability_stacked")
    os.makedirs(out_dir, exist_ok=True)

    AP_session = list(product(sessions, APs))
    with ProcessPoolExecutor(max_workers=100) as executor:
        futures = [executor.submit(process_ses_ap, ses, AP, vps, mni_dir, out_dir) for ses, AP in AP_session]
        for f in futures:
            print(f"[INFO] Finished {f.result()}")

In [ ]:
import os
import nibabel as nib
from itertools import product
import tempfile
import subprocess

vps = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 39, 40, 41, 43, 18, 19, 20, 21, 23, 24, 22, 25, 26, 27, 28, 29, 30, 31, 33, 34, 35, 36, 37, 38, 42]

def process_ses_ap(ses, AP, vps, mni_dir, out_dir):
    dictionary = {}
    n_volumes = None

    for vp in vps:
        pid = f"sub-{vp:02d}"
        subjid = f"{pid}_{ses}"
        path = os.path.join(mni_dir, f"{subjid}_{AP}_desc-MNI.nii.gz")
        dictionary[vp] = path
        if n_volumes is None:
            n_volumes = nib.load(path).shape[-1]

    print(f"[INFO] Stacking data for {ses} {AP}")
    with tempfile.TemporaryDirectory() as tmp:
        for vol in range(n_volumes):
            vol_files = []
            for vp in vps:
                vol_path = os.path.join(tmp, f"vp{vp}_vol{vol}.nii.gz")
                subprocess.run(["fslroi", dictionary[vp], vol_path, str(vol), "1"], check=True)
                print(f"{ses} | {vol_path}")
                vol_files.append(vol_path)
            out_path = os.path.join(out_dir, f"{ses}_{AP}_vol{vol}.nii.gz")
            subprocess.run(["fslmerge", "-t", out_path] + vol_files, check=True)
            #  for f in vol_files:
            #     os.remove(f)

    print(f"[INFO] Done {ses} {AP}")
    return f"{ses}_{AP}"

if __name__ == "__main__":
    mni_dir = os.path.join("/home/malberti/Unix_Folders/SWEEP2", "Gradients_Stability_Analysis", "Volume_MNI")
    out_dir = os.path.join("/home/malberti/Unix_Folders/SWEEP2", "Gradients_Stability_stacked")
    os.makedirs(out_dir, exist_ok=True)

    AP_session = list(product(sessions, APs))
    for ses, AP in AP_session:
        result = process_ses_ap(ses, AP, vps, mni_dir, out_dir)
        print(f"[INFO] Finished {result}")

# Merge the volumes together and run voxel-wise analysis: 
------------------------------------------------------------

In [ ]:
import os
import nibabel as nib
from nilearn import image as nlimage
from joblib import Parallel, delayed

vps = [i for i in range(44) if i not in [32]]
sessions = ["ses-01", "ses-02"]
APs = ["A", "B", "C"]
SMOOTH_FWHM_MM = "6"
TEMPLATE = "MNI"

home = r"/home/malberti/Unix_Folders/SWEEP2/derivatives"
out_dir = os.path.join("/home/malberti/Unix_Folders/SWEEP2", "Gradients_Stability_Analysis")

N_WORKERS = 20

def smooth_volume(session, AP, vol):
    pid = f"{session}_{AP}_vol{vol}"
    DWI_img = os.path.join(out_dir, "Volume_stacked_MNI", f"{pid}.nii")
    DWI_out_dir = os.path.join(out_dir, f"Volume_stacked_MNI{SMOOTH_FWHM_MM}mm")
    os.makedirs(DWI_out_dir, exist_ok=True)
    DWI_out = os.path.join(DWI_out_dir, f"{pid}_{SMOOTH_FWHM_MM}mm.nii")

    print(f"[INFO] Smoothing {pid}")
    smoothed = nlimage.smooth_img(DWI_img, fwhm=float(SMOOTH_FWHM_MM))
    smoothed.to_filename(DWI_out)
    return pid

if __name__ == "__main__":
    jobs = []
    for AP in APs:
        ref_path = os.path.join(home, "sub-00", "ses-01", "dwi", "signal_drift", f"sub-00_ses-01_dwi_eddy_corrected_{AP}_noPA.nii")
        n_volumes = nib.load(ref_path).shape[-1]
        for session in sessions:
            for vol in range(n_volumes):
                jobs.append((session, AP, vol))

    results = Parallel(n_jobs=N_WORKERS, backend="loky", verbose=10)(delayed(smooth_volume)(session, AP, vol) for session, AP, vol in jobs)

    print(f"[INFO] Finished smoothing {len(results)} volumes")

In [ ]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from nilearn import image as nlimage
from nilearn.glm.second_level import SecondLevelModel
from nilearn.glm import threshold_stats_img
from nilearn import plotting
import matplotlib.pyplot as plt

vps = [i for i in range(44) if i not in [32]]
sessions = ["ses-01", "ses-02"]
APs = ["A", "B", "C"]
SMOOTH_FWHM_MM = "6"
TEMPLATE = "MNI"

home = r"/home/malberti/Unix_Folders/SWEEP2/derivatives"
out_dir = os.path.join("/home/malberti/Unix_Folders/SWEEP2", "Gradients_Stability_Analysis")
smooth_dir = os.path.join(out_dir, f"Volume_stacked_MNI{SMOOTH_FWHM_MM}mm")
plot_dir = os.path.join(out_dir, "paired_ttest_plots")
os.makedirs(plot_dir, exist_ok=True)

n_subj = len(vps)
subjects = [f"sub-{vp:02d}" for vp in vps]
condition_effect = np.hstack(([1] * n_subj, [0] * n_subj))
subject_effect = np.vstack((np.eye(n_subj), np.eye(n_subj)))
paired_design_matrix = pd.DataFrame(
    np.hstack((condition_effect[:, np.newaxis], subject_effect)),
    columns=["A+B"] + subjects,
)

fig, ax = plt.subplots(1, 1, figsize=(8, 6), constrained_layout=True)
#plot_design_matrix(paired_design_matrix, rescale=False, axes=ax)
ax.set_title("Paired design", fontsize=12)

results = []

for AP in APs:
    ref_path = os.path.join(home, "sub-00", "ses-01", "dwi", "signal_drift", f"sub-00_ses-01_dwi_eddy_corrected_{AP}_noPA.nii")
    n_volumes = nib.load(ref_path).shape[-1]
    ref_bval =  np.loadtxt(os.path.join(home, "sub-00", "ses-01", "dwi", "signal_drift", f"sub-00_ses-01_dwi_eddy_corrected_{AP}_noPA.bval"))

    for vol in range(n_volumes):
        pid_ses01 = f"ses-01_{AP}_vol{vol}"
        pid_ses02 = f"ses-02_{AP}_vol{vol}"
        ses01_path = os.path.join(smooth_dir, f"{pid_ses01}_{SMOOTH_FWHM_MM}mm.nii")
        ses02_path = os.path.join(smooth_dir, f"{pid_ses02}_{SMOOTH_FWHM_MM}mm.nii")

        ses01_imgs = list(nlimage.iter_img(ses01_path))
        ses02_imgs = list(nlimage.iter_img(ses02_path))

        imgs = ses01_imgs + ses02_imgs

        model = SecondLevelModel(n_jobs=2, verbose=0).fit(imgs, design_matrix=paired_design_matrix)
        maps = model.compute_contrast("A+B", output_type="all")

        thresholded_map, threshold = threshold_stats_img(maps["z_score"], alpha=0.001, cluster_threshold=10, two_sided=True)
        results.append({"AP": AP, "vol": vol, "map": thresholded_map, "threshold": threshold})

        thresholded_map.to_filename(os.path.join(out_dir, f"paired_ttest_zmap_{AP}_vol{vol}_smooth{SMOOTH_FWHM_MM}mm.nii.gz"))

        glass_path = os.path.join(plot_dir, f"glassbrain_{AP}_vol{vol}_smooth{SMOOTH_FWHM_MM}mm.png")
        display = plotting.plot_glass_brain(
            thresholded_map,
            threshold=3.29, #threshold,
            colorbar=True,
            plot_abs=False,
            display_mode="ortho",
            title=f"{AP} vol{vol} (z>{threshold:.2f}) | BVAL = {ref_bval[vol]}",
        )

        plt.show()

## PLOT pval maps
-----------------------------------

In [ ]:
import os
import numpy as np
import nibabel as nib
from nibabel.affines import apply_affine
import matplotlib.pyplot as plt
import seaborn as sns

vps = [i for i in range(44) if i not in [32]]
APs = ["A", "B", "C"]
SMOOTH_FWHM_MM = "6"
ZTHRESH = 3.29
VLIM = 5  # fixed color scale: blue at -5, white/grey at 0, red at +5

home = r"/home/malberti/Unix_Folders/SWEEP2/derivatives"
out_dir = os.path.join("/home/malberti/Unix_Folders/SWEEP2", "Gradients_Stability_Analysis")
plot_dir = os.path.join(out_dir, "paired_ttest_plots")

for AP in APs:
    ref_path = os.path.join(home, "sub-00", "ses-01", "dwi", "signal_drift", f"sub-00_ses-01_dwi_eddy_corrected_{AP}_noPA.nii")
    n_volumes = nib.load(ref_path).shape[-1]

    rows = []
    row_labels = []
    shape, affine = None, None

    for vol in range(n_volumes):
        zpath = os.path.join(out_dir, f"paired_ttest_zmap_{AP}_vol{vol}_smooth{SMOOTH_FWHM_MM}mm.nii.gz")
        img = nib.load(zpath)
        data = img.get_fdata()
        shape, affine = data.shape, img.affine  # same grid every volume
        data[(data > -2) & (data < 2)] = 0
        row = np.ravel(data)

        if not np.any(row):
            print(f"[INFO] AP={AP} vol{vol}: entirely empty, skipping")
            continue

        rows.append(row)
        row_labels.append(f"vol{vol}")

    if not rows:
        print(f"[WARN] AP={AP}: no non-empty volumes, skipping plot")
        continue

    axcodes = nib.aff2axcodes(affine)
    print(f"[INFO] AP={AP}: orientation = {axcodes}")  # confirm A/P direction before trusting the sign below

    matrix = np.vstack(rows)

    sig_cols = np.any(np.abs(matrix) > ZTHRESH, axis=0)
    original_voxel_indices = np.where(sig_cols)[0]
    matrix = matrix[:, sig_cols]

    voxel_ijk = np.array(np.unravel_index(original_voxel_indices, shape)).T
    voxel_mni = apply_affine(affine, voxel_ijk)
    y_coords = voxel_mni[:, 1]  # anterior-posterior axis: negative = posterior, positive = anterior (if RAS+)

    sort_order = np.argsort(y_coords)  # ascending: most posterior first (left) -> most anterior last (right)
    matrix = matrix[:, sort_order]
    y_sorted = y_coords[sort_order]

    fig, (ax_strip, ax) = plt.subplots(2, 1, figsize=(10, 5), dpi=300, gridspec_kw={"height_ratios": [0.003, 1]}, constrained_layout=True,)

    strip = ax_strip.imshow(y_sorted[np.newaxis, :], aspect="auto", cmap="viridis")
    ax_strip.set_xticks([])
    ax_strip.set_yticks([])
    ax_strip.set_title(f"Z-score — AP={AP}", fontsize=20)
    fig.colorbar(strip, ax=ax_strip, orientation="horizontal", fraction=0.3, pad=0.4, label="MNI y (posterior <- 0 -> anterior)")

    sns.heatmap(
        matrix,
        cmap="seismic", center=0, vmin=-VLIM, vmax=VLIM,
        cbar_kws={"label": "Ses-01 > Ses-02"}, linecolor='black',
        xticklabels=False, yticklabels=False,
        ax=ax)
    ax.set_xlabel("Voxel, sorted posterior (left) -> anterior (right)")
    ax.set_ylabel("Volume")

    out_path = os.path.join(plot_dir, f"zmap_matrix_AP{AP}.png")
    plt.show()
    print(f"[INFO] Saved {out_path}")

print("[INFO] Done")